# Seaborn Phase 1: The Upgraded Canvas (Basics)
### Credit Card Risk Analysis Project

Seaborn is built directly on top of Matplotlib — it doesn't replace what you've learned,
it gives you higher-level functions that draw common statistical charts in one line,
while still handing you back a normal Matplotlib `Axes` you can customize exactly like
before. Everything from Phases 1-4 (titles, labels, subplots, twin axes, saving figures)
still applies; Seaborn just changes how the *chart itself* gets drawn.

This notebook covers 2 topics:
1. **Matplotlib Integration** — Seaborn draws the chart, Matplotlib finishes it
2. **Aesthetics & Themes** — setting your whole project's look in one line

**Format:** Each question has a `YOUR CODE HERE` cell to attempt first, followed by a
`Solution` cell. Try your own answer before peeking!

Run the setup cell below first.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

n = 500

age = np.clip(np.random.normal(40, 12, n), 18, 80)
housing_status = np.random.choice(['Rent', 'Own', 'Mortgage'], size=n, p=[0.35, 0.25, 0.40])
employment_type = np.random.choice(['Salaried', 'Self-Employed', 'Unemployed'],
                                    size=n, p=[0.6, 0.3, 0.1])

annual_income = np.random.lognormal(mean=10.8, sigma=0.4, size=n)
total_debt = annual_income * np.random.uniform(0.05, 0.55, size=n)
debt_to_income = (total_debt / annual_income) * 100

credit_score = np.clip(np.random.normal(680, 55, n), 300, 850)
credit_limit = np.clip(3000 + annual_income * 0.15 + np.random.normal(0, 2000, n), 500, None)

raw_risk = (debt_to_income / 100) * 0.6 + ((850 - credit_score) / 550) * 0.6
employment_bump = np.where(employment_type == 'Unemployed', 0.15, 0)
default_probability = np.clip(raw_risk + employment_bump + np.random.normal(0, 0.08, n), 0.01, 0.95)
default = np.random.binomial(1, default_probability)

risk_tier = pd.cut(credit_score, bins=[300, 600, 700, 850], labels=['High', 'Medium', 'Low'])

df = pd.DataFrame({
    'Age': age,
    'Housing_Status': housing_status,
    'Employment_Type': employment_type,
    'Annual_Income': annual_income,
    'Debt_to_Income': debt_to_income,
    'Credit_Score': credit_score,
    'Credit_Limit': credit_limit,
    'Default_Probability': default_probability,
    'Default': default,
    'Risk_Tier': risk_tier
})

# A small monthly trend series, reused later for line plots
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_volume = np.array([1200, 1350, 1280, 1450, 1600, 1550,
                            1700, 1650, 1800, 1750, 1900, 2100])

print(df.shape)
df.head()


---
## Section 1: Matplotlib Integration

Seaborn functions draw onto a Matplotlib `Axes`. If you don't give them one, they create
their own `Figure`/`Axes` behind the scenes and hand the `Axes` back to you — which means
every `ax.set_title()`, `ax.set_xlabel()`, `fig.savefig()`, etc. you already know still
works exactly the same way.


**Q1.** Plot a histogram of `df['Credit_Score']` using `sns.histplot(data=df, x='Credit_Score')`. Then, on a separate line, use Matplotlib's `plt.title("Credit Score Distribution")` to add a title — Seaborn drew the bars, Matplotlib named the chart.

In [ ]:
# YOUR CODE HERE


**Solution 1**

In [ ]:
sns.histplot(data=df, x='Credit_Score')
plt.title("Credit Score Distribution")
plt.show()


**Q2.** Plot `monthly_volume` against `months` using `sns.lineplot(x=months, y=monthly_volume)`, then use `plt.xlabel("Month")`, `plt.ylabel("Applications")`, and `plt.xticks(rotation=45)` — all plain Matplotlib calls — to finish the chart.

In [ ]:
# YOUR CODE HERE


**Solution 2**

In [ ]:
sns.lineplot(x=months, y=monthly_volume)
plt.xlabel("Month")
plt.ylabel("Applications")
plt.xticks(rotation=45)
plt.show()


**Q3.** This time use the explicit OO approach: create `fig, ax = plt.subplots()` first, then pass `ax=ax` into `sns.boxplot(data=df, x='Risk_Tier', y='Debt_to_Income', ax=ax)` so Seaborn draws directly onto *your* Axes instead of creating its own. Finish with `ax.set_title("Debt-to-Income by Risk Tier")`.

In [ ]:
# YOUR CODE HERE


**Solution 3**

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x='Risk_Tier', y='Debt_to_Income', order=['Low', 'Medium', 'High'], ax=ax)
ax.set_title("Debt-to-Income by Risk Tier")
plt.show()


**Q4.** Call `sns.scatterplot(data=df, x='Annual_Income', y='Credit_Limit')` *without* passing an `ax` argument and without creating a Figure yourself — capture its return value in a variable `ax`. Print `type(ax)` to confirm it's a normal Matplotlib `Axes`, then call `ax.set_xlabel()` and `ax.set_ylabel()` on it directly.

In [ ]:
# YOUR CODE HERE


**Solution 4**

In [ ]:
ax = sns.scatterplot(data=df, x='Annual_Income', y='Credit_Limit')
print(type(ax))
ax.set_xlabel("Annual Income")
ax.set_ylabel("Credit Limit")
plt.show()


**Q5.** Create a Figure with 1 row, 2 columns using `plt.subplots(1, 2, figsize=(12, 5))`. On the left Axes, plot `sns.histplot(data=df, x='Age', ax=axes[0])`. On the right Axes, plot `sns.boxplot(data=df, y='Debt_to_Income', ax=axes[1])`. Add one `fig.suptitle("Applicant Overview")` covering both.

In [ ]:
# YOUR CODE HERE


**Solution 5**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(data=df, x='Age', ax=axes[0])
sns.boxplot(data=df, y='Debt_to_Income', ax=axes[1])
fig.suptitle("Applicant Overview")
plt.show()


**Q6.** Compute default rate (%) per `Employment_Type`. Plot it with `sns.barplot(x=..., y=..., ax=ax)`, then call Matplotlib's `ax.bar_label(ax.containers[0], fmt='%.1f%%')` — the same `bar_label()` method from the Matplotlib phases works on Seaborn's bars too, since `ax.containers[0]` holds the bar objects Seaborn drew.

In [ ]:
# YOUR CODE HERE


**Solution 6**

In [ ]:
rate_by_employment = (df.groupby('Employment_Type')['Default'].mean() * 100).reset_index()

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(data=rate_by_employment, x='Employment_Type', y='Default', ax=ax)
ax.bar_label(ax.containers[0], fmt='%.1f%%')
ax.set_ylabel("Default Rate (%)")
plt.show()


**Q7.** Plot `sns.histplot(data=df, x='Credit_Score')` on its own Axes (capture it as `ax`), then save the figure to disk with the plain Matplotlib call `plt.savefig('credit_score_hist.png')`. Print a confirmation message after saving.

In [ ]:
# YOUR CODE HERE


**Solution 7**

In [ ]:
ax = sns.histplot(data=df, x='Credit_Score')
plt.savefig('credit_score_hist.png')
print("Saved credit_score_hist.png")
plt.show()


**Q8.** Plot `sns.lineplot(data=df.sort_values('Age'), x='Age', y='Default_Probability', hue='Housing_Status')` — multiple colored lines, one per housing group. Then use plain Matplotlib to add `ax.grid(True, alpha=0.3)` and reposition the legend with `ax.legend(title='Housing Status', loc='upper left')`.

In [ ]:
# YOUR CODE HERE


**Solution 8**

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=df.sort_values('Age'), x='Age', y='Default_Probability', hue='Housing_Status', ax=ax)
ax.grid(True, alpha=0.3)
ax.legend(title='Housing Status', loc='upper left')
plt.show()


**Q9 (Capstone).** Build one chart that leans on both libraries: `sns.scatterplot(data=df, x='Annual_Income', y='Credit_Limit', hue='Risk_Tier', ax=ax)` for the drawing, then fit a plain Matplotlib trend line with `np.polyfit` and overlay it with `ax.plot()`. Finish with `ax.set_title()`, axis labels, and `ax.grid(alpha=0.3)` — all Matplotlib.

In [ ]:
# YOUR CODE HERE


**Solution 9**

In [ ]:
slope, intercept = np.polyfit(df['Annual_Income'], df['Credit_Limit'], 1)
x_line = np.linspace(df['Annual_Income'].min(), df['Annual_Income'].max(), 100)
y_line = slope * x_line + intercept

fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=df, x='Annual_Income', y='Credit_Limit', hue='Risk_Tier',
                 hue_order=['Low', 'Medium', 'High'], alpha=0.7, ax=ax)
ax.plot(x_line, y_line, color='black', linewidth=2, label='Trend')
ax.set_title("Income vs. Credit Limit by Risk Tier", fontsize=13, fontweight='bold')
ax.set_xlabel("Annual Income")
ax.set_ylabel("Credit Limit")
ax.grid(alpha=0.3)
ax.legend()
plt.show()


> **Checkpoint — Section 1:** Seaborn functions either draw onto an `Axes` you hand them
> (`ax=ax`) or create one and return it — either way, you end up holding a normal
> Matplotlib `Axes`/`Figure`. Every finishing touch you already know (`set_title`,
> `set_xlabel`, `grid`, `legend`, `savefig`, even `bar_label`) keeps working exactly the
> same. Seaborn's real value is the *drawing* — grouped statistics, `hue=` splitting,
> and sensible defaults — not a replacement for Matplotlib.


---
## Section 2: Aesthetics & Themes

Instead of manually setting `grid()`, colors, and font sizes on every chart, Seaborn lets
you set your whole project's look with one line at the top of a notebook.


**Q10.** Call `sns.set_theme()` with no arguments (Seaborn's overall default look), then plot `sns.histplot(data=df, x='Credit_Score')`. Compare this mentally to the plain Matplotlib histograms from earlier phases — notice the background grid and softer colors that came for free.

In [ ]:
# YOUR CODE HERE


**Solution 10**

In [ ]:
sns.set_theme()
sns.histplot(data=df, x='Credit_Score')
plt.title("Credit Score Distribution (Default Theme)")
plt.show()


**Q11.** Call `sns.set_style("whitegrid")` (a clean white background with light gridlines — a common choice for financial reports since it's easy to print), then re-plot the same `Credit_Score` histogram.

In [ ]:
# YOUR CODE HERE


**Solution 11**

In [ ]:
sns.set_style("whitegrid")
sns.histplot(data=df, x='Credit_Score')
plt.title("Credit Score Distribution (whitegrid)")
plt.show()


**Q12.** `sns.set_style()` changes the style *globally* for every chart from then on. To apply a style to just *one* chart without affecting the rest of the notebook, use the context manager form: `with sns.axes_style("darkgrid"): <plot here>`. Plot the `Credit_Score` histogram inside that `with` block, then plot it again afterward with no `with` block — the second one should fall back to whatever the global style currently is (`whitegrid`, from Q11).

In [ ]:
# YOUR CODE HERE


**Solution 12**

In [ ]:
with sns.axes_style("darkgrid"):
    sns.histplot(data=df, x='Credit_Score')
    plt.title("Inside the context manager: darkgrid")
    plt.show()

sns.histplot(data=df, x='Credit_Score')
plt.title("Outside the context manager: back to whitegrid")
plt.show()


**Q13.** Create a 2x2 grid of subplots. Loop over the four built-in styles `['darkgrid', 'whitegrid', 'white', 'ticks']` together with `axes.flat`, and inside each loop iteration use `with sns.axes_style(style):` to plot `sns.histplot(data=df, x='Credit_Score', ax=ax)` — note that a `with sns.axes_style(...)` block still affects the styling of a plot drawn with an explicit `ax=`, even though the Axes already exists. Title each panel with the style's name.

In [ ]:
# YOUR CODE HERE


**Solution 13**

In [ ]:
styles = ['darkgrid', 'whitegrid', 'white', 'ticks']

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

for style, ax in zip(styles, axes.flat):
    with sns.axes_style(style):
        sns.histplot(data=df, x='Credit_Score', ax=ax)
    ax.set_title(style)

fig.tight_layout()
plt.show()


**Q14.** Call `sns.set_palette("Set2")` to switch the default color palette, then plot `sns.boxplot(data=df, x='Risk_Tier', y='Debt_to_Income', hue='Risk_Tier', order=['Low', 'Medium', 'High'])` and notice the new colors.

In [ ]:
# YOUR CODE HERE


**Solution 14**

In [ ]:
sns.set_palette("Set2")
sns.boxplot(data=df, x='Risk_Tier', y='Debt_to_Income', hue='Risk_Tier',
            order=['Low', 'Medium', 'High'], legend=False)
plt.title("Debt-to-Income by Risk Tier (Set2 palette)")
plt.show()


**Q15.** Build a custom palette with `sns.color_palette("coolwarm", 3)` (3 colors from the coolwarm colormap), store it in a variable `custom_palette`, and pass it explicitly to `sns.boxplot(..., palette=custom_palette)` for the same Risk Tier chart — this is how you apply a specific palette to one chart without changing the global default.

In [ ]:
# YOUR CODE HERE


**Solution 15**

In [ ]:
custom_palette = sns.color_palette("coolwarm", 3)

sns.boxplot(data=df, x='Risk_Tier', y='Debt_to_Income', hue='Risk_Tier',
            order=['Low', 'Medium', 'High'], palette=custom_palette, legend=False)
plt.title("Debt-to-Income by Risk Tier (custom coolwarm palette)")
plt.show()


**Q16.** Compare `sns.set_context("notebook")` (the default, sized for a Jupyter notebook) against `sns.set_context("talk")` (bigger fonts and lines, sized for a presentation slide). Create a 1x2 subplot grid, set context `"notebook"` before drawing the left histogram of `Credit_Score` and context `"talk"` before drawing the right one, keeping both on the same Figure.

In [ ]:
# YOUR CODE HERE


**Solution 16**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.set_context("notebook")
sns.histplot(data=df, x='Credit_Score', ax=axes[0])
axes[0].set_title("context='notebook'")

sns.set_context("talk")
sns.histplot(data=df, x='Credit_Score', ax=axes[1])
axes[1].set_title("context='talk'")

fig.tight_layout()
plt.show()

sns.set_context("notebook")  # reset for later cells


**Q17.** Plot the `Credit_Score` histogram, then call `sns.despine()` — this removes the top and right spines (the box lines) from the current Axes, a subtle but standard polish for a cleaner-looking report chart.

In [ ]:
# YOUR CODE HERE


**Solution 17**

In [ ]:
sns.histplot(data=df, x='Credit_Score')
plt.title("Credit Score Distribution (despined)")
sns.despine()
plt.show()


**Q18 (Capstone).** Set the project's overall look with `sns.set_theme(style='whitegrid', palette='muted')`, then build a final polished chart: `sns.boxplot(data=df, x='Risk_Tier', y='Debt_to_Income', hue='Risk_Tier', order=['Low', 'Medium', 'High'])`, followed by `sns.despine()`, a bold Matplotlib title, and axis labels.

In [ ]:
# YOUR CODE HERE


**Solution 18**

In [ ]:
sns.set_theme(style='whitegrid', palette='muted')

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='Risk_Tier', y='Debt_to_Income', hue='Risk_Tier',
            order=['Low', 'Medium', 'High'], legend=False, ax=ax)
sns.despine(ax=ax)
ax.set_title("Debt-to-Income by Risk Tier", fontsize=14, fontweight='bold')
ax.set_xlabel("Risk Tier")
ax.set_ylabel("Debt-to-Income Ratio (%)")
plt.show()


---
## Checkpoint: Seaborn Phase 1 Complete

You've covered:
- **Matplotlib integration** — Seaborn draws onto (or creates and returns) a normal
  `Axes`, so every Matplotlib finishing touch you already know — titles, labels, legends,
  grids, `bar_label()`, `savefig()` — still applies without change
- **Aesthetics & themes** — `sns.set_theme()` and `sns.set_style()` for a global look,
  `sns.axes_style()` as a context manager for a one-off style, `sns.set_palette()` /
  `sns.color_palette()` for color, `sns.set_context()` for font/line scaling by audience
  (notebook vs. talk), and `sns.despine()` for a cleaner finish

**Next up:** the statistical plot types Seaborn is really built for — things like
`sns.violinplot()`, `sns.pairplot()`, and built-in grouped comparisons (`hue=`) that
would take many more lines of raw Matplotlib. Let me know when you're ready!
